# SVM MNIST Experiments

This notebook wires up the utilities in `MNIST_models/svm_MNIST` to reproduce three tiers of experiments:

1. Baseline linear SVM in pixel space
2. Random projection variants of the linear SVM
3. Kernel SVMs inspected through their dual variables

Each block is parameterized so you can expand runs (more seeds/epochs) without changing the code structure.


In [ ]:
from MNIST_models.svm_MNIST.data_utils import load_and_project
from MNIST_models.svm_MNIST.linear_svm import LinearSVMConfig, run_multiple_seeds
from MNIST_models.svm_MNIST.kernel_svm import KernelConfig, multi_seed_kernel
from MNIST_models.svm_MNIST.workflow import (
    run_kernel_sweep,
    run_linear_sweep,
    run_projection_grid,
 )


## 1. Baseline linear SVM (pixel space)

Use a compact training loop that logs the parameter trajectory. Increase `num_epochs`, `record_interval`, or `seeds` when you want finer Hilbert distance curves or more endpoints for the spectral study.


In [ ]:
# Load the fixed 3-vs-8 split without projection
baseline_split = load_and_project(max_samples=5000, random_state=0, target_dim=None)

# Lightweight config to keep the example quick; bump epochs for full study
linear_cfg = LinearSVMConfig(lr=1e-3, weight_decay=1e-4, batch_size=256, num_epochs=2)

baseline_runs = run_multiple_seeds(
    baseline_split.X_train,
    baseline_split.y_train,
    config=linear_cfg,
    seeds=[0, 1],
 )

# Inspect Hilbert/angle/distance metrics for the first run
first_run = baseline_runs[0]
first_run.metrics


## 2. Random projection variants

Sweep over several embedding dimensions to see how the final-direction spectrum flattens. The helper returns both the individual runs and the singular values of stacked endpoints.


In [ ]:
projection_dims = [None, 256, 2000]
projection_results = run_projection_grid(
    dims=projection_dims,
    seeds=[0, 1, 2],
    base_config=linear_cfg,
    data_kwargs=dict(max_samples=4000, random_state=0),
)

for dim, result in projection_results.items():
    print(f"dim={dim}: singular values -> {result.singular_values[:5]}")


## 3. Kernel SVM dual-space check

Train RBF SVMs on different shuffles and analyze the dual coefficients as positive vectors inside the cone.


In [ ]:
kernel_cfg = KernelConfig(C=5.0, kernel="rbf", gamma="scale")
kernel_result = run_kernel_sweep(
    split=baseline_split,
    seeds=[0, 1, 2],
    config=kernel_cfg,
)

print("Dual-alpha spectrum (top 5):", kernel_result.singular_values[:5])
